# 01_02 Normalising words: stems, lemmas and the stop list that deletes "not"

"Drop", "drops", "dropped" and "dropping" are one idea written four ways. A program that counts them as
four different words learns a quarter as much from each. This notebook reduces words to one form two ways,
finds where each way goes wrong, and fixes a cleaning function that silently turns complaints into praise.

**How this notebook works.** Every notebook in this course has the same rhythm:

1. **Recall.** Answer from memory before you look anything up. `ask()` tells you at once whether you were right.
2. **Predict, then run.** Before a cell with a surprise in it, write your prediction into `guess()`. The next cell runs the code and `reveal()` compares.
3. **Worked example, then your turn.** One example is done in full; the next, near-identical one has lines marked `# YOUR CODE HERE`.
4. **Check.** A `check_...()` cell tests what you saved, exactly as the checkpoint will, and says what to fix.

Run cells in order with **Shift+Enter**. If you get lost, **Kernel, Restart Kernel and Run All Cells** starts clean.

Running this in Google Colab? This cell sets it up; in CourseLabs it does nothing.

In [ ]:
# Colab setup. In a CourseLabs session this cell does nothing.
import os, sys
if "google.colab" in sys.modules:
    import importlib, importlib.util, subprocess
    LAB, REPO = "lab-nlp-01-why-cant-a-computer-read", "/content/nlp-course"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/fenago/nlp-course.git", REPO], check=True)
    os.chdir(f"{REPO}/{LAB}")
    if not os.path.exists("data"):
        os.symlink("../data", "data")
    os.makedirs("out", exist_ok=True)
    os.environ["NLPLAB_DATA"] = f"{REPO}/data"
    sys.path.insert(0, os.getcwd())
    PIP = {'spacy': 'spacy',
           'nltk': 'nltk',
           'transformers': 'transformers',
           'en_core_web_sm': 'https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl'}
    missing = [spec for mod, spec in PIP.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
        importlib.invalidate_caches()
    import nltk
    for pkg in ['punkt_tab', 'stopwords', 'wordnet', 'omw-1.4', 'averaged_perceptron_tagger_eng', 'maxent_ne_chunker_tab', 'words']:
        nltk.download(pkg, quiet=True)
    print(f"Ready: {LAB} and its data are in {os.getcwd()}; installed {len(missing)} package(s).")
elif not os.path.isdir("/opt/nlplab/data") and os.path.isdir("data"):
    # A downloaded copy on your own computer: the helpers read data/ from here.
    os.environ["NLPLAB_DATA"] = os.path.abspath("data")

In [ ]:
import csv
import json
import os
import nltk
import spacy
from nltk.corpus import stopwords, wordnet
from nltk.stem import PorterStemmer, SnowballStemmer, LancasterStemmer, WordNetLemmatizer
from nlpcheck import ask, guess, reveal, check_01_02

nlp = spacy.load("en_core_web_sm")
tickets = list(csv.DictReader(open("data/kittiwake_tickets.csv")))
print(len(tickets), "tickets;", tickets[0]["text"])

## 1. Recall

From the last notebook.

**r3.** How many tokens does NLTK make of the word `can't`? (a number)

**r4.** Why does a subword tokenizer never meet an unknown word?
(a) it builds any word from smaller pieces it knows, (b) it skips words it does not know,
(c) its vocabulary contains every English word

In [ ]:
ask("r3", "")
ask("r4", "")

## 2. Stemming: cutting suffixes off by rule

A **stemmer** applies a list of suffix rules ("remove *-ing*", "turn *-ies* into *-i*") without knowing
what any word means. NLTK has three. Porter (1980) is the classic; Snowball is Porter's own revision;
Lancaster is the most aggressive.

Predict what Porter makes of `university` and of `universe`.

In [ ]:
guess("porter_university", None)   # a string, in quotes
guess("porter_universe", None)

In [ ]:
porter, snowball, lancaster = PorterStemmer(), SnowballStemmer("english"), LancasterStemmer()
words = ["university", "universe", "running", "runs", "ran", "hobbies", "generously",
         "general", "organization", "organ", "news", "dropped", "charges", "billing", "fairly"]
print(f"{'word':14} {'Porter':10} {'Snowball':10} {'Lancaster':10}")
for w in words:
    print(f"{w:14} {porter.stem(w):10} {snowball.stem(w):10} {lancaster.stem(w):10}")
reveal("porter_university", porter.stem("university"))
reveal("porter_universe", porter.stem("universe"))

Both become `univers`. So do `universal` and `universes`. A search for "university fees" now matches a
page about the universe: this is **overstemming**, two different meanings merged into one stem. Porter
also turns `organization` into `organ`, and Lancaster turns `news` into `new`. And `ran` is untouched by
all three, because no suffix rule turns *ran* into *run*: that needs a dictionary.

Stems are also not words: `hobbi`, `gener`, `fairli`. That is fine for a search index nobody reads, and a
problem for anything a person sees.

## 3. Lemmatization: looking the word up

A **lemmatizer** looks each word up in a dictionary (NLTK uses **WordNet**) and returns its **lemma**, the
form you would look up. It needs to know which part of speech the word is. The book's notebook called it
with no part of speech. Predict what it returns for `running`, and for `was`.

In [ ]:
lem = WordNetLemmatizer()
guess("lemma_running", None)
guess("lemma_was", None)

In [ ]:
reveal("lemma_running", lem.lemmatize("running"))
reveal("lemma_was", lem.lemmatize("was"))
print("told it is a verb:", lem.lemmatize("running", pos="v"), lem.lemmatize("was", pos="v"))

With no part of speech, `lemmatize` assumes **noun**. "Running" is a perfectly good noun (the running of
a business), so it comes back unchanged, which is the result the book printed. And "was", treated as a
plural noun, loses its *s* and becomes `wa`. Told they are verbs, both come out right: `run` and `be`.

So a lemmatizer needs a tagger in front of it. `nltk.pos_tag` gives Penn Treebank tags (`VBD`, `NNS`,
`JJ`, ...); WordNet wants one of four letters. The mapping is the first letter of the tag.

**Your turn:** finish `wordnet_pos` so that tags starting with `J` return `wordnet.ADJ`, `V` return
`wordnet.VERB`, `R` return `wordnet.ADV`, and everything else `wordnet.NOUN`.

In [ ]:
def wordnet_pos(penn_tag):
    # YOUR CODE HERE: map the first letter of the Penn tag to a WordNet part of speech
    return wordnet.NOUN

sentence = "My calls were dropping and I was charged twice"
tagged = nltk.pos_tag(nltk.word_tokenize(sentence))
print(tagged)
print([lem.lemmatize(w, wordnet_pos(t)) for w, t in tagged])

When `wordnet_pos` is right, the last line reads `['My', 'call', 'be', 'drop', 'and', 'I', 'be', 'charge',
'twice']`. As shipped it treats everything as a noun and you get `wa` again.

spaCy does all of this in one pass, because its pipeline tags before it lemmatizes:

In [ ]:
print([(t.text, t.lemma_) for t in nlp(sentence)])

## 4. Stop words, and the planted bug

**Stop words** are the very common words (*the, a, is, of*) that carry little topic on their own.
Removing them shrinks the text a program has to handle. NLTK's English list has 198 of them. Predict what
is left of this sentence after they are removed.

In [ ]:
sw = set(stopwords.words("english"))
t = "The network is not working and I don't want to pay."
print(len(sw), "stop words")
guess("after_stopwords", None)   # write the words you expect to survive, as one string

In [ ]:
left = [w for w in nltk.word_tokenize(t) if w.lower() not in sw]
reveal("after_stopwords", " ".join(left))
print("'not' in the list:", "not" in sw, "| 'no':", "no" in sw, "| \"n't\":", "n't" in sw, "| \"don't\":", "don't" in sw)

`network working n't want pay .` The complaint now reads as a network that is working. "not" is on the
list, so it went; "n't" is not on the list, because the list holds "don't" as one word and the tokenizer
has already split it, so it survived by accident. A sentiment model trained on this text learns that
"working" appears in complaints.

The function below is the cleaning step as the book's notebooks wrote it, with a lemmatizer instead of a
stemmer. It has one planted bug: `KEEP`, the set of words that must survive the stop list, is empty.
**Fix it** so that negation is kept (`not`, `no`, `nor`, `never`), then run the cell again. Note the first
line of the loop already rewrites `n't` as `not`, so one set is all it takes.

In [ ]:
KEEP = set()   # the planted bug: negation is not kept

def clean(text):
    out = []
    for word, tag in nltk.pos_tag(nltk.word_tokenize(text)):
        word = "not" if word.lower() == "n't" else word.lower()
        if not any(ch.isalnum() for ch in word):
            continue                      # punctuation
        if word in sw and word not in KEEP:
            continue                      # stop word
        out.append(lem.lemmatize(word, wordnet_pos(tag)))
    return out

for r in tickets[:4]:
    print(r["text"])
    print("  ->", clean(r["text"]))

When `KEEP` is fixed, "I'm not paying for a service that doesn't work" comes out with two `not`s in it.
When `wordnet_pos` is fixed too, `dropping` becomes `drop` and `calls` becomes `call`.

Save the cleaned versions of the first 20 tickets and check them:

In [ ]:
os.makedirs("out", exist_ok=True)
cleaned = {r["ticket_id"]: clean(r["text"]) for r in tickets[:20]}
json.dump(cleaned, open("out/01_02_clean.json", "w"), indent=1)
check_01_02()

## 5. Exit ticket

**x2.** What is the difference between a stem and a lemma? (a) a lemma is always a real word and a stem
need not be, (b) a stem is always shorter, (c) there is no difference in NLTK

In [ ]:
ask("x2", "")

Explain it back: why is removing stop words safe for a search index and dangerous for a complaint
classifier? One or two sentences.

*Your explanation:* 